In [ ]:
 import os
import glob
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Datasets/sepsis_training_data.zip"
EXTRACT_DIR = "/content/sepsis_data"       # will contain training_setA/, training_setB/

# ---------------------------------------------------------------------------
# 0a. MOUNT + EXTRACT
# ---------------------------------------------------------------------------
def mount_and_extract(drive_zip_path, extract_dir):
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists(drive_zip_path):
        raise FileNotFoundError(
            f"{drive_zip_path} not found — check the path in your Drive."
        )

    if not os.path.isdir(extract_dir):
        print(f"Extracting {drive_zip_path} -> {extract_dir}")
        with zipfile.ZipFile(drive_zip_path, "r") as z:
            z.extractall(extract_dir)
    else:
        print(f"{extract_dir} already exists, skipping extraction.")

    # Locate the two set folders wherever they landed (zip may nest them
    # one level deep depending on how it was created).
    set_a_dir = None
    set_b_dir = None
    for root, dirs, _ in os.walk(extract_dir):
        for d in dirs:
            if d.lower() == "training_seta":
                set_a_dir = os.path.join(root, d)
            elif d.lower() == "training_setb":
                set_b_dir = os.path.join(root, d)

    if set_a_dir is None or set_b_dir is None:
        raise FileNotFoundError(
            f"Could not find training_setA / training_setB under {extract_dir}. "
            f"Check the zip's internal folder names."
        )
    print(f"setA -> {set_a_dir}")
    print(f"setB -> {set_b_dir}")
    return set_a_dir, set_b_dir


# ---------------------------------------------------------------------------
# 0b. LOAD — build one long dataframe per set, tagged with source_set,
#     then concat. Keeping the tag means every downstream stat can be
#     sliced by set OR viewed combined.
# ---------------------------------------------------------------------------
def load_set(data_dir, set_label):
    files = sorted(glob.glob(os.path.join(data_dir, "*.psv")))
    print(f"[{set_label}] Found {len(files)} patient files.")
    frames = []
    for f in files:
        df = pd.read_csv(f, sep="|")
        df["patient_id"] = f"{set_label}_" + os.path.basename(f).replace(".psv", "")
        df["hour"] = np.arange(len(df))
        df["source_set"] = set_label
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


set_a_dir, set_b_dir = mount_and_extract(DRIVE_ZIP_PATH, EXTRACT_DIR)
df_a = load_set(set_a_dir, "A")
df_b = load_set(set_b_dir, "B")
df = pd.concat([df_a, df_b], ignore_index=True)

# ---------------------------------------------------------------------------
# 0c. COLUMN INVENTORY — full list, dtypes, non-null counts, and a
#     consistency check across setA/setB (and across individual files)
# ---------------------------------------------------------------------------
print("\n=== COLUMN INVENTORY ===")
print(f"Total columns: {len(df.columns)}")
for col in df.columns:
    print(f"  {col:20s} dtype={str(df[col].dtype):10s} non-null={df[col].notna().sum()}")

# Check setA and setB actually have the same columns — a silent mismatch
# here would mean concat is quietly filling gaps with NaN, which would
# corrupt your missingness numbers in Section 3.
cols_a, cols_b = set(df_a.columns), set(df_b.columns)
if cols_a != cols_b:
    print(f"\n⚠ setA and setB have DIFFERENT columns!")
    print(f"  In A but not B: {cols_a - cols_b}")
    print(f"  In B but not A: {cols_b - cols_a}")
else:
    print("\n✓ setA and setB have identical columns.")

# Check every individual .psv file matches the expected column set —
# catches the rare malformed file before it silently distorts your stats.
expected_cols = set(df.columns) - {"patient_id", "hour", "source_set"}
mismatched_files = []
for label, data_dir in [("A", set_a_dir), ("B", set_b_dir)]:
    for f in glob.glob(os.path.join(data_dir, "*.psv")):
        file_cols = set(pd.read_csv(f, sep="|", nrows=0).columns)
        if file_cols != expected_cols:
            mismatched_files.append((label, f, file_cols))

if mismatched_files:
    print(f"\n⚠ {len(mismatched_files)} file(s) have unexpected columns:")
    for label, f, cols in mismatched_files[:10]:  # cap output
        print(f"  [{label}] {os.path.basename(f)}: {cols}")
else:
    print("✓ All individual patient files match the expected column set.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/sepsis_data already exists, skipping extraction.
setA -> /content/sepsis_data/training/training_setA
setB -> /content/sepsis_data/training/training_setB
[A] Found 20336 patient files.
[B] Found 20000 patient files.

=== COLUMN INVENTORY ===
Total columns: 44
  HR                   dtype=float64    non-null=1398811
  O2Sat                dtype=float64    non-null=1349474
  Temp                 dtype=float64    non-null=525226
  SBP                  dtype=float64    non-null=1325945
  MAP                  dtype=float64    non-null=1358940
  DBP                  dtype=float64    non-null=1065656
  Resp                 dtype=float64    non-null=1313875
  EtCO2                dtype=float64    non-null=57636
  BaseExcess           dtype=float64    non-null=84145
  HCO3                 dtype=float64    non-null=65028
  FiO2                 dtype=float64    

In [ ]:
df.head()

,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id,hour,source_set
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,83.14,0,NaN,NaN,-0.03,1,0,A_p000001,0,A
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,83.14,0,NaN,NaN,-0.03,2,0,A_p000001,1,A
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,83.14,0,NaN,NaN,-0.03,3,0,A_p000001,2,A
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,83.14,0,NaN,NaN,-0.03,4,0,A_p000001,3,A
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,83.14,0,NaN,NaN,-0.03,5,0,A_p000001,4,A


# Shape

In [ ]:
print("\n=== 1. BASIC SHAPE ===")
for label, sub in [("A", df_a), ("B", df_b), ("A+B", df)]:
    n_patients = sub["patient_id"].nunique()
    n_rows = len(sub)
    print(f"[{label}] Patients: {n_patients} | Total hourly rows: {n_rows}")

print(f"\nColumns: {list(df.columns)}")

seq_lengths = df.groupby(["source_set", "patient_id"]).size()
print("\nSequence length (hours per patient) distribution, by set:")
print(seq_lengths.groupby("source_set").describe())

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, label in zip(axes, ["A", "B"]):
    seq_lengths.loc[label].hist(bins=50, ax=ax)
    ax.set_title(f"ICU stay length in hours — set{label}")
    ax.set_xlabel("Hours")
axes[0].set_ylabel("Number of patients")
plt.tight_layout()
plt.savefig("seq_length_distribution.png", dpi=120)
plt.close()
print("-> saved seq_length_distribution.png")



=== 1. BASIC SHAPE ===
[A] Patients: 20336 | Total hourly rows: 790215
[B] Patients: 20000 | Total hourly rows: 761995
[A+B] Patients: 40336 | Total hourly rows: 1552210

Columns: ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel', 'patient_id', 'hour', 'source_set']

Sequence length (hours per patient) distribution, by set:
              count       mean        std  min   25%   50%   75%    max
source_set                                                             
A           20336.0  38.857937  22.308659  8.0  25.0  39.0  47.0  336.0
B           20000.0  38.099750  23.275253  8.0  23.0  38.0  47.0  336.0
-> saved se

# Class Balance

In [ ]:
print("\n=== 2. CLASS BALANCE ===")
for label, sub in [("A", df_a), ("B", df_b), ("A+B", df)]:
    row_positive_rate = sub["SepsisLabel"].mean()
    patient_ever_positive = sub.groupby("patient_id")["SepsisLabel"].max()
    patient_positive_rate = patient_ever_positive.mean()
    print(f"[{label}] Row-level positive rate: {row_positive_rate:.4f} | "
          f"Patient-level positive rate: {patient_positive_rate:.4f} | "
          f"Positive hours: {int(sub['SepsisLabel'].sum())} | "
          f"Ever-positive patients: {int(patient_ever_positive.sum())}")
# This tells you directly whether Exp #11's spillover analysis has enough
# severe episodes to say anything within EACH set (and whether A and B
# differ enough that they shouldn't just be pooled without checking).


=== 2. CLASS BALANCE ===
[A] Row-level positive rate: 0.0217 | Patient-level positive rate: 0.0880 | Positive hours: 17136 | Ever-positive patients: 1790
[B] Row-level positive rate: 0.0141 | Patient-level positive rate: 0.0571 | Positive hours: 10780 | Ever-positive patients: 1142
[A+B] Row-level positive rate: 0.0180 | Patient-level positive rate: 0.0727 | Positive hours: 27916 | Ever-positive patients: 2932


# Missingness

In [ ]:
print("\n=== 3. MISSINGNESS ===")
non_clinical = {"Unit1", "Unit2", "Age", "Gender", "HospAdmTime", "ICULOS",
                      "SepsisLabel", "patient_id", "hour", "source_set", "n_measured", "onset_hour", "rel_hour"}

feature_cols = [c for c in df.columns if c not in non_clinical]

for label, sub in [("A", df_a), ("B", df_b), ("A+B", df)]:
    missing_pct = sub[feature_cols].isna().mean().sort_values(ascending=False)
    print(f"\n[{label}] Missingness by feature (top 10 worst):")
    print(missing_pct.head(10))
    print(f"[{label}] Overall missing ratio: {sub[feature_cols].isna().mean().mean():.4f}")

sofa_relevant = ["Platelets", "Bilirubin_total", "Creatinine", "MAP",
                  "Resp", "FiO2", "SaO2", "Glucose"]
present = [c for c in sofa_relevant if c in df.columns]
print(f"\nSOFA-relevant columns present: {present}")
for label, sub in [("A", df_a), ("B", df_b)]:
    print(f"[{label}] missingness for SOFA-relevant columns:")
    print(sub[present].isna().mean().sort_values(ascending=False))
# If A and B differ a lot here, that's a reason to report severity-tag
# results per set (or add source_set as a covariate) rather than assuming
# one pooled missingness pattern applies everywhere.


=== 3. MISSINGNESS ===

[A] Missingness by feature (top 10 worst):
EtCO2               1.000000
TroponinI           0.998779
Bilirubin_direct    0.998504
Fibrinogen          0.992369
Bilirubin_total     0.987734
Alkalinephos        0.985407
AST                 0.985042
Lactate             0.965651
PTT                 0.951525
SaO2                0.950444
dtype: float64
[A] Overall missing ratio: 0.7939

[B] Missingness by feature (top 10 worst):
HCO3                0.998148
BaseExcess          0.997685
Bilirubin_direct    0.997627
Fibrinogen          0.994472
Chloride            0.993846
PTT                 0.990298
AST                 0.982463
Alkalinephos        0.982403
Bilirubin_total     0.982352
TroponinI           0.981869
dtype: float64
[B] Overall missing ratio: 0.8089

[A+B] Missingness by feature (top 10 worst):
Bilirubin_direct    0.998074
Fibrinogen          0.993402
TroponinI           0.990477
Bilirubin_total     0.985092
Alkalinephos        0.983932
AST                

# Confound Check
This section checks whether missingness itself is predictive or not per set

In [ ]:
print("\n=== 4. MEASUREMENT DENSITY vs. SEPSIS LABEL ===")
df["n_measured"] = df[feature_cols].notna().sum(axis=1)
for label in ["A", "B", None]:
    sub = df if label is None else df[df["source_set"] == label]
    tag = "A+B" if label is None else label
    density_by_label = sub.groupby("SepsisLabel")["n_measured"].mean()
    corr = sub[["n_measured", "SepsisLabel"]].corr().iloc[0, 1]
    print(f"[{tag}] Avg non-null features/hour by label:\n{density_by_label}")
    print(f"[{tag}] Correlation(measurement density, SepsisLabel): {corr:.4f}\n")
# If this correlation holds in BOTH sets independently, confound #2 is a
# robust property of the data-collection process, not an artifact of one
# hospital's charting habits — strengthens the case for a mandatory
# covariate rather than a per-set quirk.


=== 4. MEASUREMENT DENSITY vs. SEPSIS LABEL ===
[A] Avg non-null features/hour by label:
SepsisLabel
0    6.993732
1    7.594655
Name: n_measured, dtype: float64
[A] Correlation(measurement density, SepsisLabel): 0.0194

[B] Avg non-null features/hour by label:
SepsisLabel
0    6.47971
1    7.64462
Name: n_measured, dtype: float64
[B] Correlation(measurement density, SepsisLabel): 0.0372

[A+B] Avg non-null features/hour by label:
SepsisLabel
0    6.740407
1    7.613949
Name: n_measured, dtype: float64
[A+B] Correlation(measurement density, SepsisLabel): 0.0281



# Missingness Pattern
random vs. clustered around severe episodes

In [ ]:
print("\n=== 5. MISSINGNESS AROUND SEPSIS ONSET ===")

for label, sub in [("A", df_a), ("B", df_b)]:
    ever_pos = sub.groupby("patient_id")["SepsisLabel"].max()
    septic_patients = sub[sub["patient_id"].isin(ever_pos[ever_pos == 1].index)].copy()

    # Get each patient's first onset HOUR (not row index), then map it back —
    # this replaces the old groupby.apply(hours_before_onset) entirely.
    onset_hour_map = (
        septic_patients[septic_patients["SepsisLabel"] == 1]
        .groupby("patient_id")["hour"]
        .min()
    )
    septic_patients["onset_hour"] = septic_patients["patient_id"].map(onset_hour_map)
    septic_patients["rel_hour"] = septic_patients["hour"] - septic_patients["onset_hour"]
    septic_patients["n_measured"] = septic_patients[feature_cols].notna().sum(axis=1)

    near_onset = septic_patients[(septic_patients["rel_hour"] >= -6) & (septic_patients["rel_hour"] < 0)]
    far_from_onset = septic_patients[septic_patients["rel_hour"] < -24]

    print(f"[{label}] Avg features measured, 6h before onset: "
          f"{near_onset['n_measured'].mean():.2f} (n={len(near_onset)})")
    print(f"[{label}] Avg features measured, >24h before onset: "
          f"{far_from_onset['n_measured'].mean():.2f} (n={len(far_from_onset)})\n")


=== 5. MISSINGNESS AROUND SEPSIS ONSET ===
[A] Avg features measured, 6h before onset: 7.30 (n=8894)
[A] Avg features measured, >24h before onset: 7.70 (n=58652)

[B] Avg features measured, 6h before onset: 7.13 (n=5185)
[B] Avg features measured, >24h before onset: 7.26 (n=38908)



In [ ]:
print("\n=== 6. PER-FEATURE MISSINGNESS AROUND SEPSIS ONSET (exploratory screen) ===")
# NOTE: exploratory / uncorrected for multiple comparisons — used to select
# candidate features for the embedding capacity test, not a standalone claim.
# Onset definition (first SepsisLabel==1 hour) is identical to Section 5.

MIN_N = 200  # floor below which we don't trust a per-window estimate

results = []



for label, sub in [("A", df_a), ("B", df_b), ("A+B", df)]:
    ever_pos = sub.groupby("patient_id")["SepsisLabel"].max()
    septic_patients = sub[sub["patient_id"].isin(ever_pos[ever_pos == 1].index)].copy()

    onset_hour_map = (
        septic_patients[septic_patients["SepsisLabel"] == 1]
        .groupby("patient_id")["hour"]
        .min()
    )
    septic_patients["onset_hour"] = septic_patients["patient_id"].map(onset_hour_map)
    septic_patients["rel_hour"] = septic_patients["hour"] - septic_patients["onset_hour"]

    near_onset = septic_patients[(septic_patients["rel_hour"] >= -6) & (septic_patients["rel_hour"] < 0)]
    far_from_onset = septic_patients[septic_patients["rel_hour"] < -24]

    for feat in feature_cols:
        near_measured = near_onset[feat].notna()
        far_measured = far_from_onset[feat].notna()

        n_near = near_measured.sum()          # rows with the feature measured
        n_far = far_measured.sum()
        n_near_total = len(near_onset)        # total rows in window (denominator)
        n_far_total = len(far_from_onset)

        # patient-level counts — guards against one chatty patient inflating n
        patients_near = near_onset.loc[near_measured, "patient_id"].nunique()
        patients_far = far_from_onset.loc[far_measured, "patient_id"].nunique()

        rate_near = near_measured.mean() if n_near_total > 0 else np.nan
        rate_far = far_measured.mean() if n_far_total > 0 else np.nan

        sufficient = (n_near >= MIN_N) and (n_far >= MIN_N)

        results.append({
            "source_set": label,
            "feature": feat,
            "rate_near_onset": rate_near,
            "rate_far_onset": rate_far,
            "diff": rate_near - rate_far if sufficient else np.nan,
            "n_rows_near": n_near_total,
            "n_rows_far": n_far_total,
            "n_patients_near": patients_near,
            "n_patients_far": patients_far,
            "sufficient_n": sufficient,
        })

per_feature_missingness = pd.DataFrame(results).sort_values(
    ["source_set", "diff"], key=lambda s: s.abs() if s.name == "diff" else s,
    ascending=False, na_position="last"
)

per_feature_missingness.to_csv("per_feature_missingness_screen.csv", index=False)
print(per_feature_missingness.head(20))
print(f"\nSaved full table to per_feature_missingness_screen.csv "
      f"({(~per_feature_missingness['sufficient_n']).sum()} feature-set rows flagged insufficient_n)")


=== 6. PER-FEATURE MISSINGNESS AROUND SEPSIS ONSET (exploratory screen) ===
   source_set     feature  rate_near_onset  rate_far_onset      diff  \
41          B       EtCO2         0.244744        0.274828 -0.030083   
40          B        Resp         0.766056        0.790274 -0.024219   
55          B     Glucose         0.257666        0.274417 -0.016750   
44          B        FiO2         0.046866        0.053794 -0.006928   
59          B   Potassium         0.099325        0.104786 -0.005461   
51          B     Calcium         0.085246        0.090444 -0.005198   
56          B     Lactate         0.038573        0.042742 -0.004169   
46          B       PaCO2         0.047637        0.051558 -0.003920   
34          B          HR         0.909547        0.913283 -0.003736   
57          B   Magnesium         0.054966        0.058497 -0.003531   
45          B          pH         0.047830        0.051352 -0.003522   
47          B        SaO2         0.039923        0.042356 

In [ ]:
# ---------------------------------------------------------------------------
# 6. RESOLUTION STARTER — what to actually do about what you find
# ---------------------------------------------------------------------------
print("\n=== 6. NEXT STEPS BASED ON WHAT YOU JUST SAW ===")
print("""
- If setA and setB differ meaningfully on missingness, class balance, or
  the onset-density pattern: treat source_set as a modeling covariate (or
  at minimum report results split by set), not just a loading artifact.
  It's kept as a column in `df` for exactly this reason.

- If SOFA-relevant columns are >80-90% missing per hour (in either set):
  don't build a strict per-hour SOFA score. Use forward-fill plus a binary
  "was this measured this hour" indicator feature.

- If row-level positive rate is under ~2% (in either set):
  don't use raw accuracy anywhere. Use AUPRC or F1 on the positive class.

- If measurement density correlates with SepsisLabel in BOTH sets (step 4):
  confound #2 is confirmed and generalizes — treat it as a mandatory
  covariate in the severity~attention regression.

- If absolute count of severe hours is small in one or both sets (step 2):
  Experiment #11's spillover analysis may be underpowered. Consider
  pooling A+B for that specific analysis (with source_set as a covariate)
  or the MIMIC-IV/eICU upgrade path.

- Sequence length distribution (step 1), viewed per set, tells you your
  max useful Transformer context window per set and whether A and B need
  different padding strategies.
""")


=== 6. NEXT STEPS BASED ON WHAT YOU JUST SAW ===

- If setA and setB differ meaningfully on missingness, class balance, or
  the onset-density pattern: treat source_set as a modeling covariate (or
  at minimum report results split by set), not just a loading artifact.
  It's kept as a column in `df` for exactly this reason.

- If SOFA-relevant columns are >80-90% missing per hour (in either set):
  don't build a strict per-hour SOFA score. Use forward-fill plus a binary
  "was this measured this hour" indicator feature.

- If row-level positive rate is under ~2% (in either set):
  don't use raw accuracy anywhere. Use AUPRC or F1 on the positive class.

- If measurement density correlates with SepsisLabel in BOTH sets (step 4):
  confound #2 is confirmed and generalizes — treat it as a mandatory
  covariate in the severity~attention regression.

- If absolute count of severe hours is small in one or both sets (step 2):
  Experiment #11's spillover analysis may be underpowered. Consider